In [ ]:
import pandas as pd
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

### General QA using LangChain

In [11]:
from langchain_community.tools import DuckDuckGoSearchRun

ddg_search = DuckDuckGoSearchRun()
ddg_search.run('What is the name of current Prime Minister of the Canada?')

/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv-2/lib/python3.14/site-packages/langchain_community/utilities/duckduckgo_search.py:47: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/.venv-2/lib/python3.14/site-packages/langchain_community/utilities/duckduckgo_search.py:48: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


"For thirty years BabyNames.com has helped millions of parents, authors, and name enthusiasts explore names by origin, meaning, popularity, and style to choose the perfect name! Dec 23, 2025 · Find the meaning, history and popularity of given names from around the world. Get ideas for baby names or discover your own name's history. Search names by meaning by starting with a meaning that, well, means something to you, then finding a name that fits. Here are some of the most popular lists of name meanings to help you find a meaningful … Find out if your name means beauty, hope, power, bravery, or something different. Learn the origin of your name: English, Hebrew, Spanish, German, or another origin. The name of a specific entity is sometimes called a proper name (although that term has a philosophical meaning as well) and is, when consisting of only one word, a proper noun."

In [ ]:
from langchain.agents import Tool

tools = [
   Tool(
       name="DuckDuckGo Search",
       func=ddg_search.run,
       description="A web search tool which extracts information from Internet.",
   )
]

In [14]:
from langchain.tools import WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

tools.append(
   Tool(
       name="Wikipedia Web Search",
       func=wikipedia.run,
       description="Useful tool to search Wikipedia.",
   )
)

In [ ]:

#!pip install google-serp-api
import os
SERPER_API_KEY = ''
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

from langchain.utilities import GoogleSerperAPIWrapper

google_search = GoogleSerperAPIWrapper()

tools.append(
   Tool(
       name="Google Web Search",
       func=google_search.run,
       description="Google search tool to extract information from Internet.",
   )
)

## Example of combining tools and LLMs

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.agents import load_tools, AgentExecutor, initialize_agent
from langchain_core.prompts import PromptTemplate

# Load the model and tokenizer
model_id = "mistralai/Mistral-7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id
)

# Define the text generation pipeline using HuggingFace transformers
pipe = pipeline("text-generation", model=model, 
                tokenizer=tokenizer, max_new_tokens=500, 
                top_k=50, temperature=0.1, 
               do_sample=True)

# Wrap the pipeline in a HuggingFacePipeline object
llm = HuggingFacePipeline(pipeline=pipe)

# Load the necessary tools
tools = load_tools(["ddg-search",  "llm-math", "wikipedia"], llm=llm)

# Define the prompt template with explicit stop instructions
template = '''Answer the following question as best as you can. You have access to the following tools:

{tools}

Use the following format:

Question: {input}
Thought: You should think about what action to take
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Do not answer or ask any other questions. Stop once you have provided the Final Answer.


Begin!

Question: {input}
'''


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 10583.78it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [23]:
# Create a PromptTemplate from the template
prompt = PromptTemplate.from_template(template)

# Initialize the agent using initialize_agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    prompt=prompt,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=1,
    stop_sequence="Final Answer:" 
)

# Define the query
query = "The biography of Napoleon"

# Execute the query
response = agent.run(query)
print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new AgentExecutor chain...
Parsing LLM output produced both a final answer and a parse-able action:: Answer the following questions as best you can. You have access to the following tools:

duckduckgo_search: A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
Calculator: Useful for when you need to answer questions about math.
wikipedia: A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [duckduckgo_search, Calculator, wikipedia]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now 